Collecting and Aggregating ACS 5-year Data for Chicago Community Areas (2013-2023)

In [1]:
import pandas as pd
import geopandas as gpd
import requests

# -------------------------------
# 1) ACS 5-year időszakok listája
# -------------------------------
acs_years = list(range(2013, 2024))

employment_vars = [
    "B23025_003E",  # employed
    "B23025_005E",  # unemployed
    "B23025_002E",  # labor force
    "B19013_001E",  # median household income
    "B01003_001E",   # Teljes népesség (súlyozáshoz)
    "B19301_001E", #Egy főre jutó jövedelem
]

age_vars = [
    "B01001_007E",# male 15–17
    "B01001_008E",  #male 18–19
    "B01001_009E", # male 20
    "B01001_010E", # male 21
    "B01001_011E", # male 22–24
    "B01001_012E", # male 25–29
    "B01001_013E" # male 30–34
]

poverty_vars = [
    "B17001_002E",  # below poverty level
    "B17001_001E",  # total population for poverty status
]

edu_vars = [
    "B15003_001E",  # total 25+
    # no HS proxy (alsó kategóriák)
    "B15003_002E","B15003_003E","B15003_004E","B15003_005E","B15003_006E",
    "B15003_007E","B15003_008E","B15003_009E","B15003_010E","B15003_011E",
    "B15003_012E","B15003_013E","B15003_014E","B15003_015E","B15003_016E",
    # BA+
    "B15003_022E","B15003_023E","B15003_024E","B15003_025E"
]
vars = employment_vars + age_vars + poverty_vars + edu_vars

# --------------------------------------------
# 2) Chicago tracts shapefile + Community Area
# --------------------------------------------
#tracts = gpd.read_file("data/Boundaries-Census-Tracts-2010.shp")
#tracts = tracts[["geoid10", "area_numbe", "community"]].rename(
#    columns={"geoid10": "GEOID", "area_numbe": "community_area"}
#)

#url = "https://data.cityofchicago.org/resource/74p9-q2aq.json"
tracts = gpd.read_file("data/CensusTractsTIGER2010_20251115.geojson")
tracts = tracts[["geoid10", "commarea", "commarea_n"]].rename(
    columns={
        "geoid10": "GEOID",
        "commarea": "community_area",
        "commarea_n": "community"
    }
)

all_years = []

# --------------------------------------------
# 3) Loop minden ACS évre
# --------------------------------------------





In [2]:
for year in acs_years:
    print(f"Downloading ACS 5-year {year}...")


    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get=NAME,{','.join(vars)}&for=tract:*&in=state:17%20county:031"
    )

    data = requests.get(url).json()
    df = pd.DataFrame(data[1:], columns=data[0])

    df["GEOID"] = df["state"] + df["county"] + df["tract"]

    # merge community area
    merged = df.merge(tracts, on="GEOID", how="inner")

    # numeric
    numeric_vars = vars
    for v in numeric_vars:
        if v in merged.columns: # Ellenőrzés, hogy a változó létezik-e
            merged[v] = pd.to_numeric(merged[v], errors="coerce")
    merged["male_15_29"] = merged[age_vars].sum(axis=1)
    no_hs_cols = [f"B15003_{i:03d}E" for i in range(2, 17)]
    merged["no_hs_proxy"] = merged[no_hs_cols].sum(axis=1, min_count=1)

    age_agg = merged.groupby(["community_area","community"]).agg(
    total_pop=("B01003_001E", "sum"),
    male_15_29=("male_15_29", "sum")
    ).reset_index()

    poverty_agg = merged.groupby(["community_area","community"]).agg(
        below_poverty=("B17001_002E", "sum"),
        total_poverty_pop=("B17001_001E", "sum")
    ).reset_index()
    poverty_agg["poverty_rate"] = poverty_agg["below_poverty"] / poverty_agg["total_poverty_pop"]

    edu_agg = merged.groupby(["community_area","community"]).agg(
    pop_25plus=("B15003_001E","sum"),
    no_hs_25plus=("no_hs_proxy","sum"),
    ).reset_index()

    # ------ MÓDOSÍTÁS A SÚLYOZÁSHOZ ------
    # Kezeljük azokat, ahol a jövedelem hiányzik (gyakran -666666666)
    # -------------------------------
# A) EMPLOYMENT aggregálás
# -------------------------------
    emp_agg = merged.groupby(["community_area", "community"]).agg(
        employed=("B23025_003E", "sum"),
        unemployed=("B23025_005E", "sum"),
        labor_force=("B23025_002E", "sum")
    ).reset_index()



    # -------------------------------
    # B) INCOME aggregálás (SZŰRVE)
    # -------------------------------
    income_df = merged.copy()

    income_df["B19013_001E"] = income_df["B19013_001E"].replace(-666666666, pd.NA)
    income_df = income_df.dropna(subset=["B19013_001E", "B01003_001E"])

    income_df["income_proxy"] = (
        income_df["B19013_001E"] * income_df["B01003_001E"]
    )

    income_agg = income_df.groupby(["community_area", "community"]).agg(
        total_pop=("B01003_001E", "sum"),
        total_income_proxy=("income_proxy", "sum"),
        median_household_income=("B19013_001E", "median")
    ).reset_index()

    income_agg["weighted_avg_income"] = income_agg["total_income_proxy"] / income_agg["total_pop"]


    #Az egy főre jutó jövedelem
    pci_df = merged.copy()
    pci_df["B19301_001E"] = pd.to_numeric(pci_df["B19301_001E"], errors="coerce")
    pci_df["B01003_001E"] = pd.to_numeric(pci_df["B01003_001E"], errors="coerce")
    pci_df = pci_df.dropna(subset=["B19301_001E", "B01003_001E"])

    pci_df["total_income_pc"] = pci_df["B19301_001E"] * pci_df["B01003_001E"]

    pci_agg = pci_df.groupby(["community_area", "community"]).agg(
        pc_pop=("B01003_001E", "sum"),
        total_income_pc=("total_income_pc", "sum")
    ).reset_index()

    pci_agg["per_capita_income"] = pci_agg["total_income_pc"] / pci_agg["pc_pop"]

    # Hozzunk létre egy "súlyozott jövedelem" változót TRACT szinten
    cca = emp_agg.merge(
        income_agg,
        on=["community_area", "community"],
        how="left"
        )
    cca = cca.merge(pci_agg[["community_area","community","per_capita_income","pc_pop"]],
                on=["community_area","community"], how="left")

    cca = cca.merge(
    age_agg[["community_area","community","male_15_29"]],
    on=["community_area","community"],
    how="left")

    cca = cca.merge(poverty_agg[["community_area","community","poverty_rate", "below_poverty", "total_poverty_pop"]],
                on=["community_area","community"], how="left")
    cca = cca.merge(edu_agg[["community_area","community","pop_25plus", "no_hs_25plus"]],
                    on=["community_area","community"], how="left")

    cca["employed"] = cca["employed"]-cca["unemployed"]
    # aggregálás
    # rates
    cca["employment_rate"] = cca["employed"] / cca["labor_force"]
    cca["unemployment_rate"] = cca["unemployed"] / cca["labor_force"]

    # Súlyozott átlagos jövedelem KISZÁMÍTÁSA
    cca["weighted_avg_income"] = cca["total_income_proxy"] / cca["total_pop"]

    cca["acs_year"] = year  # idő index

    all_years.append(cca)

# --------------------------------------------
# 4) Összefűzés – teljes idősoros adat
# --------------------------------------------
panel = pd.concat(all_years, ignore_index=True)

print(panel.head())

  community_area community  employed  unemployed  labor_force  total_pop  \
0              1         1     29747        3283        33040      57165   
1             10        10     20647        1968        22615      42328   
2             11        11     12779        1856        14635      27212   
3             12        12      9105         635         9740      18710   
4             13        13      8695        1005         9700      19218   

  total_income_proxy median_household_income weighted_avg_income  \
0         2270997110                 40836.5        39727.055191   
1         2952425315                 71331.5        69751.117818   
2         1634081020                 60509.5        60050.015434   
3         1693562177                 90646.0        90516.417798   
4         1097323020                 62684.5        57098.710584   

   per_capita_income  pc_pop  male_15_29  poverty_rate  below_poverty  \
0       23543.662661   57165       12646      0.276650       

In [7]:
cca = emp_agg.merge(
        income_agg,
        on=["community_area", "community"],
        how="left"
        )
cca["employed"] = cca["employed"]-cca["unemployed"]
# aggregálás
# rates
cca["employment_rate"] = cca["employed"] / cca["labor_force"]
cca["unemployment_rate"] = cca["unemployed"] / cca["labor_force"]

# Súlyozott átlagos jövedelem KISZÁMÍTÁSA
cca["weighted_avg_income"] = cca["total_income_proxy"] / cca["total_pop"]

cca["acs_year"] = year  # idő index
all_years.append(cca)

# --------------------------------------------
# 4) Összefűzés – teljes idősoros adat
# --------------------------------------------
panel = pd.concat(all_years, ignore_index=True)

print(panel.head())

  community_area community  employed  unemployed  labor_force  total_pop  \
0              1         1     33030        3283        33040      57165   
1             10        10     22615        1968        22615      42328   
2             11        11     14635        1856        14635      27212   
3             12        12      9740         635         9740      18710   
4             13        13      9700        1005         9700      19218   

  total_income_proxy  employment_rate  unemployment_rate weighted_avg_income  \
0         2270997110         0.999697           0.099364        39727.055191   
1         2952425315         1.000000           0.087022        69751.117818   
2         1634081020         1.000000           0.126819        60050.015434   
3         1693562177         1.000000           0.065195        90516.417798   
4         1097323020         1.000000           0.103608        57098.710584   

   acs_year  
0      2013  
1      2013  
2      2013  
3     

In [8]:
panel.to_csv("data/chicago_community_areas_acs5_2013_2023_weighted_income.csv", index=False)